# BFR Algorithm

## Learning Objectives

1. **Explain** why standard k-means fails when data does not fit in RAM
2. **Define** the three BFR sets: Discard Set (DS), Compression Set (CS), Retained Set (RS)
3. **Derive** the sufficient statistics (N, SUM, SUMSQ) and how they replace raw points
4. **Apply** the Mahalanobis distance criterion for cluster assignment
5. **Implement** BFR for multi-chunk disk-resident data


## Problem Statement

### Clustering at Scale

K-means requires all data in RAM to compute centroids. For datasets with billions of points, this is impossible. BFR (Bradley-Fayyad-Reina) extends k-means to run in a single pass through disk-resident data by maintaining *compressed summaries* of clusters.

### BFR Assumption

Clusters are **Gaussian with axis-aligned covariances** — each feature is independent within a cluster. This means the cluster shape is an ellipsoid aligned with the coordinate axes (no rotated ellipsoids).

With this assumption, we can test whether a new point belongs to a cluster using the Mahalanobis distance, without storing any of the original cluster points.

### Sufficient Statistics

For a cluster with $N$ points in $d$ dimensions, we store only:
- $N$ — count of points
- $\text{SUM}[j]$ — sum of $j$-th coordinates, for $j = 1, \ldots, d$
- $\text{SUMSQ}[j]$ — sum of squared $j$-th coordinates

This gives us the centroid and per-dimension variance in $O(d)$ space, regardless of $N$.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="380" font-family="monospace" font-size="12">
  <rect width="820" height="380" fill="#fafafa" rx="8"/>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">BFR Algorithm: Three Sets for Disk-Resident Clustering</text>

  <!-- DS box -->
  <rect x="20" y="40" width="230" height="140" rx="6" fill="#e8f0fb" stroke="#4e79a7" stroke-width="2"/>
  <text x="135" y="60" text-anchor="middle" fill="#4e79a7" font-size="12" font-weight="bold">Discard Set (DS)</text>
  <text x="135" y="80" text-anchor="middle" fill="#333" font-size="11">Points assigned to a cluster</text>
  <text x="135" y="98" text-anchor="middle" fill="#333" font-size="11">with high confidence.</text>
  <text x="135" y="118" text-anchor="middle" fill="#555" font-size="11">Kept as 3 statistics:</text>
  <text x="135" y="136" text-anchor="middle" fill="#333" font-size="11">N  (count)</text>
  <text x="135" y="154" text-anchor="middle" fill="#333" font-size="11">SUM  (sum of coords)</text>
  <text x="135" y="172" text-anchor="middle" fill="#333" font-size="11">SUMSQ  (sum of coord²)</text>

  <!-- CS box -->
  <rect x="295" y="40" width="230" height="140" rx="6" fill="#fff4e0" stroke="#f28e2b" stroke-width="2"/>
  <text x="410" y="60" text-anchor="middle" fill="#f28e2b" font-size="12" font-weight="bold">Compression Set (CS)</text>
  <text x="410" y="80" text-anchor="middle" fill="#333" font-size="11">Small clusters too far from</text>
  <text x="410" y="98" text-anchor="middle" fill="#333" font-size="11">any main cluster centroid.</text>
  <text x="410" y="118" text-anchor="middle" fill="#555" font-size="11">Also kept as (N, SUM, SUMSQ)</text>
  <text x="410" y="140" text-anchor="middle" fill="#333" font-size="11">Merged with DS when</text>
  <text x="410" y="158" text-anchor="middle" fill="#333" font-size="11">close enough at end.</text>

  <!-- RS box -->
  <rect x="570" y="40" width="230" height="140" rx="6" fill="#e8f8e8" stroke="#59a14f" stroke-width="2"/>
  <text x="685" y="60" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Retained Set (RS)</text>
  <text x="685" y="80" text-anchor="middle" fill="#333" font-size="11">Isolated points, not close</text>
  <text x="685" y="98" text-anchor="middle" fill="#333" font-size="11">to any cluster or CS group.</text>
  <text x="685" y="118" text-anchor="middle" fill="#555" font-size="11">Stored as raw points.</text>
  <text x="685" y="138" text-anchor="middle" fill="#333" font-size="11">Re-evaluated each</text>
  <text x="685" y="156" text-anchor="middle" fill="#333" font-size="11">new chunk of data.</text>

  <!-- Statistics -->
  <rect x="20" y="200" width="780" height="80" rx="5" fill="#f5f5f5" stroke="#ccc" stroke-width="1.5"/>
  <text x="410" y="220" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Sufficient Statistics (per cluster, per dimension d)</text>
  <text x="410" y="242" text-anchor="middle" fill="#333" font-size="11">centroid[i] = SUM[i] / N        variance[i] = SUMSQ[i]/N − (SUM[i]/N)²</text>
  <text x="410" y="262" text-anchor="middle" fill="#555" font-size="11">These d+2d+1 numbers replace all N raw points  →  O(d) memory per cluster</text>
  <text x="410" y="278" text-anchor="middle" fill="#555" font-size="10">BFR assumption: clusters are Gaussian with axis-aligned covariance (each dim independent)</text>

  <!-- Mahalanobis -->
  <rect x="20" y="294" width="780" height="74" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="410" y="314" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">Mahalanobis Distance — Cluster Assignment Criterion</text>
  <text x="410" y="334" text-anchor="middle" fill="#333" font-size="12">MD(x, cluster) = √Σᵢ ((xᵢ − centroidᵢ) / σᵢ)²</text>
  <text x="410" y="352" text-anchor="middle" fill="#555" font-size="11">Assign x to DS if MD &lt; threshold (e.g. 2σ or 3σ). Ensures point is within cluster ellipsoid.</text>
  <text x="410" y="362" text-anchor="middle" fill="#555" font-size="10">With Gaussian assumption: 68% points within 1σ, 95% within 2σ, 99.7% within 3σ</text>
</svg>
'''
display(SVG(svg))


## Derivation

### From Statistics to Centroid and Variance

$$\text{centroid}[j] = \frac{\text{SUM}[j]}{N}$$

$$\text{variance}[j] = \frac{\text{SUMSQ}[j]}{N} - \left(\frac{\text{SUM}[j]}{N}\right)^2 = E[X_j^2] - (E[X_j])^2$$

This is just the computational formula for variance: $\text{Var}[X] = E[X^2] - (E[X])^2$.

### Mahalanobis Distance

For the Gaussian cluster assumption (independent dimensions):

$$\text{MD}(x, C) = \sqrt{\sum_{j=1}^{d} \left(\frac{x_j - \mu_j}{\sigma_j}\right)^2}$$

where $\mu_j = \text{centroid}[j]$ and $\sigma_j = \sqrt{\text{variance}[j]}$.

Assign point $x$ to the Discard Set of cluster $C$ if $\text{MD}(x, C) < \tau$ (e.g. $\tau = 3$).

### Merging Statistics

To merge two clusters $(N_1, \text{SUM}_1, \text{SUMSQ}_1)$ and $(N_2, \text{SUM}_2, \text{SUMSQ}_2)$:
$$N = N_1 + N_2, \quad \text{SUM} = \text{SUM}_1 + \text{SUM}_2, \quad \text{SUMSQ} = \text{SUMSQ}_1 + \text{SUMSQ}_2$$

This $O(d)$ operation replaces storing all the raw points.


## Algorithm Steps

1. **Initialise:** Load first chunk, run k-means in memory, build DS statistics for each cluster
2. **For each subsequent chunk:**
   - For each point $x$: find closest DS cluster by Mahalanobis distance; if $\text{MD} < \tau$: add to DS; else: add to retained set
   - Run mini-k-means on retained set to form CS groups; merge close CS groups to DS
3. **Final pass:** assign all remaining RS/CS points to nearest DS cluster
4. **Output:** $k$ cluster statistics (centroid, variance, size)


In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import List


@dataclass
class ClusterStats:
    """Sufficient statistics for one BFR cluster (or CS mini-cluster)."""
    n: int = 0
    s: np.ndarray = field(default_factory=lambda: np.array([]))   # SUM
    ss: np.ndarray = field(default_factory=lambda: np.array([]))  # SUMSQ
    cluster_id: int = -1

    def add_point(self, x: np.ndarray):
        if self.n == 0:
            self.s = x.copy()
            self.ss = x ** 2
        else:
            self.s += x
            self.ss += x ** 2
        self.n += 1

    def merge(self, other: 'ClusterStats'):
        self.n  += other.n
        self.s  += other.s
        self.ss += other.ss

    @property
    def centroid(self):
        return self.s / self.n

    @property
    def variance(self):
        return self.ss / self.n - (self.s / self.n) ** 2

    @property
    def std(self):
        return np.sqrt(np.maximum(self.variance, 1e-10))

    def mahalanobis(self, x):
        """Mahalanobis distance from x to this cluster."""
        return float(np.sqrt(np.sum(((x - self.centroid) / self.std) ** 2)))


def bfr(chunks, k, threshold_sigma=3.0):
    """
    BFR Algorithm for clustering disk-resident data.

    Inputs
    ------
    chunks          : list of np.ndarray shape (n_i, d) — data chunks loaded from disk one at a time
    k               : int — number of clusters
    threshold_sigma : float — Mahalanobis distance threshold for DS assignment

    Output
    ------
    cluster_stats : list of ClusterStats — one per cluster
    """
    from sklearn.cluster import KMeans  # only for initial clustering; could replace

    # ── Initialise: run k-means on first chunk in memory ─────────────────────
    first_chunk = chunks[0]
    km = KMeans(n_clusters=k, n_init=5, random_state=42).fit(first_chunk)

    ds = [ClusterStats(cluster_id=j) for j in range(k)]
    for x, label in zip(first_chunk, km.labels_):
        ds[label].add_point(x)

    retained = []  # raw points not yet assigned

    # ── Process remaining chunks ──────────────────────────────────────────────
    for chunk in chunks[1:]:
        new_retained = []

        for x in chunk:
            # Find closest DS cluster by Mahalanobis distance
            dists = [d.mahalanobis(x) for d in ds]
            best  = int(np.argmin(dists))

            if dists[best] < threshold_sigma:
                ds[best].add_point(x)
            else:
                new_retained.append(x)

        retained.extend(new_retained)

        # Try to form CS from retained points using k-means on retained set
        if len(retained) > 2 * k:
            arr = np.array(retained)
            km2 = KMeans(n_clusters=min(k, len(retained) // 2), n_init=3, random_state=0).fit(arr)
            for x, label in zip(arr, km2.labels_):
                ds[label % k].add_point(x)  # simplified: merge small clusters to nearest DS
            retained = []

    # ── Final pass: assign retained points to nearest DS ─────────────────────
    for x in retained:
        dists = [d.mahalanobis(x) for d in ds]
        ds[int(np.argmin(dists))].add_point(x)

    return ds


# ── Demo ──────────────────────────────────────────────────────────────────────
rng = np.random.default_rng(7)
k_true = 3
n_per_cluster = 1000

# Generate Gaussian clusters
centers = np.array([[0, 0], [5, 0], [2.5, 4]])
data = np.vstack([
    rng.normal(centers[i], 0.8, size=(n_per_cluster, 2))
    for i in range(k_true)
])
rng.shuffle(data)

# Simulate disk chunks (load 300 points at a time)
chunk_size = 300
chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

try:
    cluster_stats = bfr(chunks, k=k_true)
    print(f"BFR completed on {len(data)} points in {len(chunks)} chunks")
    for i, cs in enumerate(cluster_stats):
        print(f"  Cluster {i}: n={cs.n:4d}  centroid={cs.centroid.round(2)}")
except ImportError:
    print("sklearn not available; BFR requires KMeans for initialization")
    print("Install with: pip install scikit-learn")
